In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# 1. Load Raw Titanic Data
df = pd.read_csv('../Titanic Logistic Regression/data/train.csv')

# 2. Feature Engineering (2 New Features)
# Feature 1: FamilySize (Siblings/Spouses + Parents/Children + Self)
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Feature 2: IsAlone (1 if traveling alone, 0 otherwise)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# 3. Define Features and Target
target = 'Survived'
features = ['Pclass', 'Sex', 'Age', 'Fare', 'Embarked', 'FamilySize', 'IsAlone']

X = df[features]
y = df[target]

# 4. Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Define Transformers for Numerical & Categorical Features
num_features = ['Age', 'Fare', 'FamilySize']
cat_features = ['Pclass', 'Sex', 'Embarked', 'IsAlone']

# Numeric Pipeline: Impute missing with median -> Scale
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical Pipeline: Impute missing with mode -> One-Hot Encode
cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# 6. Bundle with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ]
)

# 7. Create Full ML Pipeline (Preprocessing + Model)
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

# 8. Train the entire pipeline in ONE call
full_pipeline.fit(X_train, y_train)

# 9. Evaluate on Test Set in ONE call
y_pred = full_pipeline.predict(X_test)

print("=== Complete Pipeline Classification Report ===")
print(classification_report(y_test, y_pred))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")

# 10. Save the Entire Pipeline to Disk
joblib.dump(full_pipeline, 'titanic_pipeline_model.joblib')
print("\n Pipeline saved successfully as 'titanic_pipeline_model.joblib'!")

=== Complete Pipeline Classification Report ===
              precision    recall  f1-score   support

           0       0.82      0.90      0.86       110
           1       0.81      0.68      0.74        69

    accuracy                           0.82       179
   macro avg       0.81      0.79      0.80       179
weighted avg       0.82      0.82      0.81       179

Accuracy: 0.8156
F1-Score: 0.7402

 Pipeline saved successfully as 'titanic_pipeline_model.joblib'!


In [4]:
loaded_pipeline = joblib.load('titanic_pipeline_model.joblib')

# mock raw passenger data
sample_passenger = pd.DataFrame([{
    'Pclass': 1,
    'Sex': 'female',
    'Age': 24,
    'Fare': 80.0,
    'Embarked': 'C',
    'FamilySize': 2,
    'IsAlone': 0
}])

prediction = loaded_pipeline.predict(sample_passenger)
probability = loaded_pipeline.predict_proba(sample_passenger)

print(f"Prediction: {'Survived' if prediction[0] == 1 else 'Did Not Survive'}")
print(f"Survival Probability: {probability[0][1] * 100:.2f}%")

Prediction: Survived
Survival Probability: 95.44%


### Task 7 Summary: ML Pipelines & Feature Engineering

#### What is a Machine Learning Pipeline?
A **Pipeline** bundles preprocessing steps (handling missing values, feature scaling, categorical encoding) and the estimator (model) into a single, unified execution object.

#### Why Pipelines Matter in Production:
1. **Prevents Data Leakage:** Statistics (e.g., median for imputation, mean/std for scaling) are learned *only* from the training set during `.fit()` and safely applied to test or incoming real-world data during `.predict()`.
2. **Eliminates Code Duplication:** No need to manually duplicate 15+ lines of data transformation logic when deploying to production or predicting on single records.
3. **Reproducibility & Modularity:** Makes model tuning, cross-validation, and hyperparameter searches completely reproducible with `ColumnTransformer`.

#### Feature Engineering Impact:
* **`FamilySize`** (`SibSp + Parch + 1`): Captured passenger group dynamics and evacuation priority.
* **`IsAlone`** (`FamilySize == 1`): Explicitly identified solo travelers with distinct survival probabilities.

#### Model Persistence:
* Exported the complete trained pipeline via `joblib.dump()` as `titanic_pipeline_model.joblib`.
* The saved artifact accepts raw, un-transformed inputs and handles end-to-end imputation, encoding, scaling, and inference seamlessly.